In [ ]:
# [auto] project-root setup
import os, sys
from pathlib import Path

# Walk upward to find the project root (folder containing .gitignore)
_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / '.gitignore').exists():
    _p = _p.parent
PROJECT_ROOT = _p

# Switch cwd to the project root so every relative path (Stage1_Exploration/, Refined_Results_v4/, ...) keeps working
os.chdir(PROJECT_ROOT)
# Let the notebooks do `from viz_config import VizConfig`
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import r2_score
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from viz_config import VizConfig

# ==========================================
# 0. Global configuration and style setup
# ==========================================
# Apply the shared VizConfig so fonts, DPI, etc. stay consistent
VizConfig.set_style()

# Pull font-size constants from VizConfig
TITLE_SIZE = VizConfig.TITLE_SIZE
LABEL_SIZE = VizConfig.LABEL_SIZE
TICK_SIZE = VizConfig.TICK_SIZE
LEGEND_SIZE = VizConfig.LEGEND_SIZE

OUTPUT_DIR = "Refined_Results_v4"
# Input path: predictions before and after MLP denoising
INPUT_PATH = os.path.join(OUTPUT_DIR, "Noise_50pct", "predictions_comparison.csv")
# Pre-defined unit string for labels
UNIT_C = r"($10^{-7} \cdot \text{kg/m}^3$)" 
UNIT_D = r"($\text{m}$)"

# ==========================================
# 1. Load and prepare data
# ==========================================
# Fall back to synthetic data if the real file is missing
if not os.path.exists(INPUT_PATH) and not os.path.exists(os.path.dirname(INPUT_PATH)):
    print("Warning: File missing - generating synthetic data for demo purposes...")
    # Synthetic: a noisy curve plus its denoised counterpart
    dist = np.linspace(0, 1000, 200)
    # Ground truth
    clean = 10 * np.exp(-dist/300) + 2 * np.sin(dist/50)
    # Noisy input
    noisy = clean + np.random.normal(0, 1, size=len(dist))
    # Predicted (MLP output - synthetic denoising)
    pred = clean + np.random.normal(0, 0.2, size=len(dist))
    
    case_df = pd.DataFrame({
        'Distance': dist, 'True_Clean': clean, 'True_Noisy': noisy, 'Pred_MLP': pred, 'Case': 'dp0'
    })
    r2_noisy, r2_denoised = 0.85, 0.98
    selected_case = "dp0"
else:
    if not os.path.exists(INPUT_PATH):
        print(f"ERROR: data file not found: {INPUT_PATH}")
        exit()
    # Load real data
    df = pd.read_csv(INPUT_PATH)
    selected_case = "dp0" # Pick a representative case
    case_df = df[df['Case'] == selected_case].sort_values('Distance')
    # Compute R^2
    r2_noisy = r2_score(case_df['True_Clean'], case_df['True_Noisy'])
    r2_denoised = r2_score(case_df['True_Clean'], case_df['Pred_MLP'])

# ==========================================
# 2. Begin plotting (1x2 subplots)
# ==========================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# ------------------------------------------
# Sub-panel (a): local sequence denoising
# ------------------------------------------
# Purpose: visualise sequence shape before/after denoising

# 1. Plot noisy data points
# Use the secondary colour (grey) so they sit in the background
ax1.scatter(case_df['Distance'], case_df['True_Noisy'], color=VizConfig.COLOR_SECONDARY, alpha=0.5, 
            s=20, label='Noisy Data (50%)', rasterized=True, edgecolors='none')

# 2. Plot ground-truth points
# Use COLOR_MAIN (deep blue) for ground truth
ax1.scatter(case_df['Distance'], case_df['True_Clean'], color=VizConfig.COLOR_MAIN, alpha=0.8,  s=20,
         label='Ground Truth (CFD)', rasterized=True, edgecolors='none')

# 3. Plot the denoised prediction curve (MLP output)
# Use the highlight colour (solid red) to emphasise the model output
ax1.plot(case_df['Distance'], case_df['Pred_MLP'], color=VizConfig.COLOR_HIGHLIGHT, linestyle='-', 
         linewidth=2.5, label='MLP Denoised Output')

# Axis labels and title
ax1.set_xlabel(f"Distance {UNIT_D}", fontsize=LABEL_SIZE, labelpad=10)
ax1.set_ylabel(f"Concentration {UNIT_C}", fontsize=LABEL_SIZE, labelpad=10)
ax1.set_title(f"(a) Local Sequence Denoising (Case: {selected_case})", fontsize=TITLE_SIZE, pad=15, fontweight='bold')
ax1.tick_params(labelsize=TICK_SIZE)
ax1.legend(fontsize=LEGEND_SIZE, loc='upper right', frameon=True)
ax1.grid(True, alpha=0.4, linestyle='--')

# ==========================================
# [Key feature] Add an inset zoom
# ==========================================
# Create a sub-region inside ax1 to show detail
# [0.45, 0.45, 0.35, 0.35] Values are [x, y, width, height] as fractions of the parent axes
axins = ax1.inset_axes([0.45, 0.45, 0.35, 0.35]) 

# Redraw the same data in the inset, keeping colours consistent
axins.scatter(case_df['Distance'], case_df['True_Noisy'], color=VizConfig.COLOR_SECONDARY, alpha=0.5, s=20, rasterized=True, edgecolors='none')
axins.scatter(case_df['Distance'], case_df['True_Clean'], color=VizConfig.COLOR_MAIN, alpha=0.8, s=20, rasterized=True, edgecolors='none')
axins.plot(case_df['Distance'], case_df['Pred_MLP'], color=VizConfig.COLOR_HIGHLIGHT, linestyle='-', linewidth=2.5)

# Set the zoom-in X range
x1, x2 = 200, 400
axins.set_xlim(x1, x2)

# Auto-compute the corresponding Y range (avoids hard-coding)
mask_zoom = (case_df['Distance'] >= x1) & (case_df['Distance'] <= x2)
y_data_segment = pd.concat([
    case_df.loc[mask_zoom, 'True_Noisy'],
    case_df.loc[mask_zoom, 'True_Clean'],
    case_df.loc[mask_zoom, 'Pred_MLP']
])

if not y_data_segment.empty:
    y_min, y_max = y_data_segment.min(), y_data_segment.max()
    y_margin = (y_max - y_min) * 0.1 
    axins.set_ylim(y_min - y_margin, y_max + y_margin)

# Tweak sub-axes styling
axins.tick_params(labelsize=10)
axins.grid(True, alpha=0.3, linestyle=':')
# Colour the inset spines
axins.spines['bottom'].set_color(VizConfig.COLOR_AXIS)
axins.spines['left'].set_color(VizConfig.COLOR_AXIS)

# Connector lines between main axes and inset (mark_inset)
mark_inset(ax1, axins, loc1=2, loc2=4, fc="none", ec="0.5", linestyle='--')

# ------------------------------------------
# Sub-panel (b): parity plot
# ------------------------------------------
# Purpose: compare "raw-noise vs ground truth"  and  "denoised vs ground truth"  distributions

# 1. Plot raw noisy data (noisy vs truth)
# Colour: grey (secondary) - shows raw-data spread
ax2.scatter(case_df['True_Clean'], case_df['True_Noisy'], color=VizConfig.COLOR_SECONDARY, alpha=0.5, 
            s=15, label=f'Raw Noise vs. Truth ($R^2 = {r2_noisy:.4f}$)', rasterized=True, edgecolors='none')

# 2. Plot denoised data (denoised vs truth)
# Colour: red (highlight) - shows denoised tightness
ax2.scatter(case_df['True_Clean'], case_df['Pred_MLP'], color=VizConfig.COLOR_HIGHLIGHT, alpha=0.6, 
            s=15, label=f'Denoised vs. Truth ($R^2 = {r2_denoised:.4f}$)', rasterized=True, edgecolors='none')

# 3. Draw the identity line (y = x)
all_vals = pd.concat([case_df['True_Clean'], case_df['True_Noisy'], case_df['Pred_MLP']])
lower_bound = all_vals.min()
upper_bound = all_vals.max()

ax2.plot([lower_bound, upper_bound], [lower_bound, upper_bound], 
         color=VizConfig.COLOR_AXIS, linestyle='--', alpha=0.8, linewidth=1.5, zorder=3)

# Axis labels and title
ax2.set_xlabel(f"Ground Truth Concentration {UNIT_C}", fontsize=LABEL_SIZE, labelpad=10)
ax2.set_ylabel(f"Observed/Predicted Concentration {UNIT_C}", fontsize=LABEL_SIZE, labelpad=10)
ax2.set_title(f"(b) Parity Plot (Case: {selected_case})", fontsize=TITLE_SIZE, pad=15, fontweight='bold')
ax2.tick_params(labelsize=TICK_SIZE)
ax2.legend(fontsize=LEGEND_SIZE, loc='upper left')
ax2.grid(True, alpha=0.4, linestyle='--')

# ==========================================
# 3. Save and export
# ==========================================
plt.tight_layout()

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

output_pdf = os.path.join(OUTPUT_DIR, "5.pdf")
plt.savefig(output_pdf, dpi=VizConfig.DPI, bbox_inches='tight', pad_inches=0.1) 

print(f"Figure saved to: {output_pdf}")
plt.show()